# Week 5: DQN 交互式教程 (DQN Interactive Tutorial)

> **课程:** CST8509 Reinforcement Learning | **主题:** Deep Q-Network with Stable-Baselines3
>
> **核心问题：** 当状态空间太大、Q-Table 装不下时，如何用神经网络替代表格来逼近 Q 值？
>
> **学习路径：** Q-Table 的瓶颈 → DQN 四大组件 → 动作空间适配 → 训练配置 → 实战演示

---

## 0. 环境准备 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 中文字体设置（如果可用）
try:
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

print("✅ 环境准备完成")

## 1. 为什么需要 DQN？(Why DQN?)

### 回顾：Q-Table 的工作方式

Week 2 学的 Q-Learning 用一张表格存储每个 (state, action) 对的价值：

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

这在小环境中完美运行。但当环境变大...

### ❌ Q-Table 的致命问题：维度灾难

| 环境规模 | 状态数量级 | Q-Table 可行？ |
|----------|-----------|---------------|
| 2×4      | ~几百     | ✅ 轻松       |
| 4×4      | ~几万     | ⚠️ 勉强       |
| 5×5      | ~几十万   | ❌ 内存爆炸   |
| 10×10    | ~天文数字 | ❌ 完全不可能 |

> 💡 **核心矛盾：** Q-Table 需要为每个状态分配一行，状态空间一大就存不下、学不完。

运行下面的代码，直观感受这个问题 👇

In [ ]:
# Demo 1: Q-Table vs DQN — State Space Explosion
env_sizes = ['2x4', '4x4', '5x5', '6x6', '8x8', '10x10']
state_counts = [24, 4_096, 120_000, 1_200_000, 100_000_000, 10_000_000_000]
qtable_memory_mb = [s * 4 * 8 / 1e6 for s in state_counts]
dqn_params = 64 * 64 + 64 * 4
dqn_memory_mb = [dqn_params * 8 / 1e6] * len(env_sizes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71' if s < 10000 else '#f39c12' if s < 1e6 else '#e74c3c' for s in state_counts]
bars = ax1.bar(env_sizes, state_counts, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_yscale('log')
ax1.set_ylabel('State Count'); ax1.set_xlabel('Environment Size')
ax1.set_title('State Space Grows Exponentially')
ax1.axhline(y=1e6, color='red', linestyle='--', alpha=0.5, label='Q-Table limit (~1M)')
ax1.legend(fontsize=9)
for bar, count in zip(bars, state_counts):
    label = f'{count/1e9:.0f}B' if count >= 1e9 else f'{count/1e6:.0f}M' if count >= 1e6 else f'{count/1e3:.0f}K' if count >= 1e3 else str(count)
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.5, label, ha='center', va='bottom', fontsize=9, fontweight='bold')

x = np.arange(len(env_sizes)); width = 0.35
ax2.bar(x - width/2, qtable_memory_mb, width, label='Q-Table', color='#e74c3c', alpha=0.8)
ax2.bar(x + width/2, dqn_memory_mb, width, label='DQN (~4K params)', color='#3498db', alpha=0.8)
ax2.set_yscale('log'); ax2.set_xticks(x); ax2.set_xticklabels(env_sizes)
ax2.set_ylabel('Memory (MB)'); ax2.set_xlabel('Environment Size')
ax2.set_title('Memory Requirement Comparison'); ax2.legend(fontsize=10)

fig.suptitle('Why DQN? Q-Table Curse of Dimensionality', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout(); plt.show()

## 2. DQN 四大组件 (DQN Four Components)

DQN 的核心思想：**用神经网络替代 Q-Table**。但直接替换会导致训练不稳定，所以 DeepMind (2015) 引入了三个关键技巧：

| 组件 | 作用 | 解决什么问题 |
|------|------|-------------|
| **Q-Network** | 输入状态 → 输出所有动作的 Q 值 | 替代 Q-Table |
| **Target Network** | Q-Network 的缓慢更新副本 | 防止"追自己尾巴"（训练不稳定） |
| **Replay Buffer** | 存储过去的经验 (s, a, r, s') | 打破样本相关性 |
| **ε-Greedy** | 以概率 ε 随机探索 | 平衡探索与利用 |

### DQN 目标 Q 值公式

$$y = r + \gamma \max_{a'} Q_{target}(s', a')$$

⚠️ 注意：用的是 **Target Network**（不是主网络）来计算目标！

### 损失函数

$$L(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( Q_\theta(s_i, a_i) - y_i \right)^2$$

运行下面的代码，可视化这四个组件 👇

In [ ]:
# Demo 2: DQN Four Components Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2a: Q-Network
ax = axes[0, 0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Q-Network: State -> Q-values', fontsize=11, fontweight='bold')
for i, label in enumerate(['s1', 's2', 's3', 's4']):
    ax.add_patch(plt.Circle((1.5, 6.5 - i*1.5), 0.4, color='#3498db', alpha=0.8))
    ax.text(1.5, 6.5 - i*1.5, label, ha='center', va='center', fontsize=9, color='white')
for i in range(3):
    ax.add_patch(plt.Circle((5, 6 - i*2), 0.4, color='#9b59b6', alpha=0.8))
    ax.text(5, 6 - i*2, f'h{i+1}', ha='center', va='center', fontsize=9, color='white')
for i, label in enumerate(['Q(a1)', 'Q(a2)', 'Q(a3)']):
    ax.add_patch(plt.Circle((8.5, 5.5 - i*2), 0.4, color='#e74c3c', alpha=0.8))
    ax.text(8.5, 5.5 - i*2, label, ha='center', va='center', fontsize=7, color='white')
for i in range(4):
    for j in range(3):
        ax.plot([1.9, 4.6], [6.5-i*1.5, 6-j*2], 'gray', alpha=0.2, linewidth=0.5)
for i in range(3):
    for j in range(3):
        ax.plot([5.4, 8.1], [6-i*2, 5.5-j*2], 'gray', alpha=0.2, linewidth=0.5)
ax.text(1.5, 0.5, 'Input: State', ha='center', fontsize=9, color='#3498db')
ax.text(8.5, 0.5, 'Output: Q-values', ha='center', fontsize=9, color='#e74c3c')

# 2b: Target Network
ax = axes[0, 1]
np.random.seed(42)
steps = np.arange(100)
no_target = np.cumsum(np.random.randn(100) * 0.5) + 5
with_target = np.zeros(100)
ct = 5.0
for i in range(100):
    if i % 20 == 0: ct = no_target[i] + np.random.randn() * 0.3
    with_target[i] = ct + np.random.randn() * 0.1
ax.plot(steps, no_target, 'r-', alpha=0.7, label='No Target Net (drifting)', linewidth=1.5)
ax.plot(steps, with_target, 'b-', alpha=0.7, label='With Target Net (stable)', linewidth=1.5)
for i in range(0, 100, 20): ax.axvline(x=i, color='blue', linestyle=':', alpha=0.3)
ax.set_xlabel('Training Steps'); ax.set_ylabel('Target Q-value')
ax.set_title('Target Network Stabilizes Training', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')

# 2c: Replay Buffer
ax = axes[1, 0]
np.random.seed(123)
seq = np.array([1,1,1,2,2,2,3,3,3,4,4,4,5,5,5])
shuf = np.random.permutation(seq)
xp = np.arange(len(seq))
ax.bar(xp - 0.2, seq, 0.35, color=plt.cm.Set1(seq/6), edgecolor='black', linewidth=0.5, label='Sequential')
ax.bar(xp + 0.2, shuf, 0.35, color=plt.cm.Set1(shuf/6), edgecolor='black', linewidth=0.5, label='Random')
ax.set_xlabel('Sample Index'); ax.set_ylabel('Env Region')
ax.set_title('Replay Buffer Breaks Correlation', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# 2d: Epsilon Decay
ax = axes[1, 1]
ts = 100000; ds = int(ts * 0.1)
sa = np.arange(ts)
eps = np.where(sa < ds, 1.0 - 0.95 * sa / ds, 0.05)
ax.plot(sa, eps, 'g-', linewidth=2); ax.fill_between(sa, eps, alpha=0.1, color='green')
ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='eps_final=0.05')
ax.axvline(x=ds, color='orange', linestyle='--', alpha=0.5, label=f'Decay ends @ {ds:,}')
ax.set_xlabel('Training Steps'); ax.set_ylabel('Epsilon')
ax.set_title('Epsilon-Greedy Decay', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle('DQN Four Core Components', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout(); plt.show()

## 3. DQN 训练 6 步流程 (DQN Training Process)

```
Step 1: 交互收集    →  Agent ↔ Env → (s, a, r, s') → Buffer
Step 2: 预热        →  Random actions × learning_starts steps
Step 3: 采样        →  Random mini-batch from Buffer
Step 4: 计算目标    →  y = r + γ max Q_target(s', a')  ← Target Network!
Step 5: 更新主网络  →  Loss = MSE(Q_main(s,a), y), backprop
Step 6: 同步目标网络 → Q_target ← Q_main (every N steps)
        ↓
        🔄 Repeat until total_timesteps done
```

### 🧮 手算练习

**已知：** r = 1, γ = 0.99, Q_target(s', a0) = 2.5, Q_target(s', a1) = 3.0, Q_target(s', a2) = 1.8

**求目标 Q 值：**

$$y = r + \gamma \max_{a'} Q_{target}(s', a') = 1 + 0.99 \times 3.0 = 3.97$$

## 4. 动作空间适配 (Action Space Adaptation)

### ⚠️ DQN 只支持 Discrete 动作空间

DQN 输出层是每个动作一个 Q 值节点，所以动作数必须有限。

但 BlocksWorld 环境使用 **MultiDiscrete** 动作空间 → 需要用 Wrapper 展平！

**核心思路：** `MultiDiscrete([2, 3])` → 总共 2×3 = 6 种组合 → `Discrete(6)`

运行下面的代码看展平过程 👇

In [ ]:
# Demo 3: MultiDiscrete -> Discrete Flattening
dims = (3, 4)
print(f"MultiDiscrete({list(dims)}) -> Discrete({np.prod(dims)})")
print(f"\nMapping:")
for i in range(dims[0]):
    for j in range(dims[1]):
        flat = i * dims[1] + j
        print(f"  ({i},{j}) -> {flat}", end="")
        restored = np.unravel_index(flat, dims)
        assert restored == (i, j)
    print()

print(f"\nnp.unravel_index examples:")
for flat_idx in [0, 5, 7, 11]:
    multi = np.unravel_index(flat_idx, dims)
    print(f"  np.unravel_index({flat_idx}, {dims}) = {multi}")

## 5. SB3 DQN 实战代码 (SB3 DQN in Practice)

### 关键超参数

| 参数 | 值 | 含义 |
|------|-----|------|
| `policy` | `"MultiInputPolicy"` | 支持字典观测 |
| `learning_starts` | `100` | 预热步数 |
| `batch_size` | `512` | Mini-batch 大小 |
| `log_interval` | `1` | TensorBoard 每 episode 记录 |
| `check_freq` | `10000` | Callback 每 10K 步触发 |

### 推理注意事项

- `deterministic=True`：推理时不探索
- VecEnv `step()` 返回 **4 个值**（不是 5 个）：`obs, reward, done, info`

## 6. DQN 训练演示：CartPole (Live Demo)

让我们在 CartPole-v1 上实际训练一个 DQN，观察学习曲线。

> CartPole 的 "solved" 阈值是平均奖励 ≥ 195。最大奖励 = 500。

In [ ]:
# Demo 5: DQN on CartPole-v1
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("CartPole-v1")
model = DQN(
    "MlpPolicy", env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    target_update_interval=500,
    verbose=0
)

eval_rewards, eval_steps = [], []
for step in range(0, 30000, 2000):
    model.learn(total_timesteps=2000, reset_num_timesteps=False)
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
    eval_rewards.append(mean_reward)
    eval_steps.append(step + 2000)
    print(f"  Step {step + 2000:>6}: reward = {mean_reward:.1f} +/- {std_reward:.1f}")

env.close()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eval_steps, eval_rewards, 'b-o', markersize=4, linewidth=2, label='DQN Mean Reward')
ax.fill_between(eval_steps, [r-20 for r in eval_rewards], [r+20 for r in eval_rewards], alpha=0.2, color='blue')
ax.axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Max Reward (500)')
ax.axhline(y=195, color='orange', linestyle='--', alpha=0.5, label='Solved Threshold (195)')
ax.set_xlabel('Training Steps'); ax.set_ylabel('Mean Reward')
ax.set_title('DQN Learning Curve on CartPole-v1', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal reward: {eval_rewards[-1]:.1f}")
print(f"Best reward: {max(eval_rewards):.1f} at step {eval_steps[eval_rewards.index(max(eval_rewards))]}")

## 7. 试一试 (Try It Yourself!)

修改下面的超参数，观察对训练效果的影响：

1. **`learning_rate`**: 试试 `1e-2`（太大）和 `1e-5`（太小）
2. **`buffer_size`**: 试试 `1000`（太小）和 `100000`（更大）
3. **`batch_size`**: 试试 `16` 和 `256`
4. **`target_update_interval`**: 试试 `100`（频繁同步）和 `5000`（很少同步）
5. **`exploration_fraction`**: 试试 `0.5`（慢衰减）和 `0.01`（快衰减）

> 💡 **提示：** 每次只改一个参数，对比学习曲线的变化！

In [ ]:
# Try modifying these parameters!
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("CartPole-v1")

model = DQN(
    "MlpPolicy", env,
    learning_rate=1e-3,          # Try: 1e-2, 1e-5
    buffer_size=50000,           # Try: 1000, 100000
    learning_starts=1000,
    batch_size=64,               # Try: 16, 256
    gamma=0.99,
    target_update_interval=500,  # Try: 100, 5000
    exploration_fraction=0.1,    # Try: 0.5, 0.01
    verbose=0
)

model.learn(total_timesteps=20000)
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20)
print(f"Result: {mean_reward:.1f} +/- {std_reward:.1f}")
env.close()

---

## 📋 本周要点回顾

| 概念 | 要点 |
|------|------|
| **Q-Table 瓶颈** | 大状态空间下内存爆炸、无法泛化 |
| **DQN 核心** | 用神经网络逼近 Q 值，固定大小参数 |
| **Target Network** | 冻结副本作为评分标准，防止目标漂移 |
| **Replay Buffer** | 随机采样打破时间相关性 |
| **ε-Greedy** | ε 从 1.0 衰减到 0.05，探索→利用 |
| **DiscreteActionWrapper** | MultiDiscrete 展平为 Discrete |
| **DQN 公式** | $y = r + \gamma \max_{a'} Q_{target}(s', a')$ |
| **VecEnv 注意** | `step()` 返回 4 个值，不是 5 个 |